In [50]:
import re
import os
from collections import Counter
import numpy as np


In [51]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens = text.split()
    return tokens

In [52]:
def build_vocabulary(texts, max_vocab_size=5000):

    all_tokens = []
    for text in texts:
        tokens = tokenizer(text)
        all_tokens.extend(tokens)
    
    freq = Counter(all_tokens)
    most_common = freq.most_common(max_vocab_size - 2)  # -2 برای <pad> و <unk>
    
    word2idx = {'<pad>': 0, '<unk>': 1}
    idx2word = ['<pad>', '<unk>']
    
    for word, _ in most_common:
        word2idx[word] = len(idx2word)
        idx2word.append(word)
    
    return word2idx, idx2word

In [53]:
def text_to_sequence(text, word2idx, max_len=None):

    tokens = tokenizer(text)
    seq = [word2idx.get(token, 1) for token in tokens] 
    
    if max_len is not None:
        if len(seq) > max_len:
            seq = seq[:max_len]
        else:
            seq = seq + [0] * (max_len - len(seq))
    
    return seq

In [54]:
def load_rt_polarity_data(data_dir):
    texts = []
    labels = []
    
    pos_path = os.path.join(data_dir, '../data/rt-polarity.pos')
    with open(pos_path, 'r', encoding='latin-1') as f:
        pos_texts = f.readlines()
        texts.extend(pos_texts)
        labels.extend([1] * len(pos_texts))
    
    neg_path = os.path.join(data_dir, '../data/rt-polarity.neg')
    with open(neg_path, 'r', encoding='latin-1') as f:
        neg_texts = f.readlines()
        texts.extend(neg_texts)
        labels.extend([0] * len(neg_texts))
    
    return texts, labels

In [55]:
from sklearn.model_selection import train_test_split

def split_data(texts, labels, test_size=0.2, val_size=0.1, random_state=42):
    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=test_size + val_size, 
        stratify=labels, random_state=random_state
    )
    
    val_ratio = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=test_size/(test_size+val_size),
        stratify=y_temp, random_state=random_state
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test

In [56]:
def report_length_stats(texts, name):
    lengths = [len(tokenizer(text)) for text in texts]
    mean_len = np.mean(lengths)
    std_len = np.std(lengths)
    print(f"{name}: mean = {mean_len:.2f}, std = {std_len:.2f}, min = {min(lengths)}, max = {max(lengths)}")
    return mean_len, std_len

In [57]:
data_dir = './data'

texts, labels = load_rt_polarity_data(data_dir)
print(f"Total samples: {len(texts)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

X_train, X_val, X_test, y_train, y_val, y_test = split_data(texts, labels)

print(f"\nTraining samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

print(f"\nClass distribution in training: Positive={sum(y_train)}, Negative={len(y_train)-sum(y_train)}")
print(f"Class distribution in validation: Positive={sum(y_val)}, Negative={len(y_val)-sum(y_val)}")
print(f"Class distribution in test: Positive={sum(y_test)}, Negative={len(y_test)-sum(y_test)}")

word2idx, idx2word = build_vocabulary(X_train, max_vocab_size=5000)
print(f"\nVocabulary size: {len(word2idx)}")

print("\n--- Sequence length statistics (token count) ---")
report_length_stats(X_train, "Training")
report_length_stats(X_val, "Validation")
report_length_stats(X_test, "Test")

Total samples: 10662
Positive: 5331, Negative: 5331

Training samples: 7463
Validation samples: 1066
Test samples: 2133

Class distribution in training: Positive=3732, Negative=3731
Class distribution in validation: Positive=533, Negative=533
Class distribution in test: Positive=1066, Negative=1067

Vocabulary size: 5000

--- Sequence length statistics (token count) ---
Training: mean = 18.61, std = 8.61, min = 1, max = 51
Validation: mean = 18.34, std = 8.81, min = 1, max = 44
Test: mean = 18.20, std = 8.40, min = 1, max = 48


(np.float64(18.204406938584153), np.float64(8.39962042806537))

### Part 2


In [58]:
import torch
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=None):
        self.texts = texts
        self.labels = labels
        self.word2idx = word2idx
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        sequence = text_to_sequence(text, self.word2idx, max_len=self.max_len)
        return torch.tensor(sequence, dtype=torch.long), torch.tensor(label, dtype=torch.float32)

In [59]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, PackedSequence

def dynamic_padding_collate(batch):

    sequences, labels = zip(*batch)    
    # Get actual lengths before padding
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    # Pad sequences to max length in this batch
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, torch.tensor(labels, dtype=torch.float32), lengths

In [60]:
class VanillaRNN(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=1):
        super(VanillaRNN, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = torch.nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True, nonlinearity='tanh')
        self.fc = torch.nn.Linear(hidden_dim, 1)
        self.sigmoid = torch.nn.Sigmoid()
    
    def forward(self, x, lengths):
        # x: (batch, seq_len)
        embedded = self.embedding(x)  # (batch, seq_len, embedding_dim)
        # Pack the padded sequence
        packed_embedded = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed_embedded)
        # hidden: (num_layers, batch, hidden_dim)
        last_hidden = hidden[-1]  # Take last layer's hidden state
        output = self.fc(last_hidden)
        output = self.sigmoid(output).squeeze()
        return output

In [61]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=1):
        super(LSTMModel, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, 1)
        self.sigmoid = torch.nn.Sigmoid()
    
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed_embedded = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed_embedded)
        last_hidden = hidden[-1]  # (batch, hidden_dim)
        output = self.fc(last_hidden)
        output = self.sigmoid(output).squeeze()
        return output

In [62]:
class GRUModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128, num_layers=1):
        super(GRUModel, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = torch.nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, 1)
        self.sigmoid = torch.nn.Sigmoid()
    
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed_embedded = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.gru(packed_embedded)
        last_hidden = hidden[-1]
        output = self.fc(last_hidden)
        output = self.sigmoid(output).squeeze()
        return output

In [63]:
def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, word2idx, batch_size=32, max_len=None):
    train_dataset = SentimentDataset(X_train, y_train, word2idx, max_len)
    val_dataset = SentimentDataset(X_val, y_val, word2idx, max_len)
    test_dataset = SentimentDataset(X_test, y_test, word2idx, max_len)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=dynamic_padding_collate)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=dynamic_padding_collate)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=dynamic_padding_collate)
    
    return train_loader, val_loader, test_loader

In [64]:
def test_models():
    vocab_size = 5000
    batch_size = 4
    seq_len = 10
    
    dummy_x = torch.randint(0, vocab_size, (batch_size, seq_len))
    dummy_lengths = torch.tensor([10, 8, 6, 5])
    
    rnn_model = VanillaRNN(vocab_size)
    rnn_model.eval()
    with torch.no_grad():
        out_rnn = rnn_model(dummy_x, dummy_lengths)
    print(f"Vanilla RNN output shape: {out_rnn.shape}")
    
    lstm_model = LSTMModel(vocab_size)
    lstm_model.eval()
    with torch.no_grad():
        out_lstm = lstm_model(dummy_x, dummy_lengths)
    print(f"LSTM output shape: {out_lstm.shape}")
    
    gru_model = GRUModel(vocab_size)
    gru_model.eval()
    with torch.no_grad():
        out_gru = gru_model(dummy_x, dummy_lengths)
    print(f"GRU output shape: {out_gru.shape}")
    

test_models()

Vanilla RNN output shape: torch.Size([4])
LSTM output shape: torch.Size([4])
GRU output shape: torch.Size([4])
